# Agent runs on the Kaggle-hosted competition model

Serves `gemma-4-31b-it-qat-w4a16-ct` with vLLM on the attached GPUs (as in the official getting-started notebook) and runs agent config k0 through the official `swegemma` harness.

Code: https://github.com/TechNaom/gemma4-agentic-swe

In [ ]:
import glob, os, subprocess, sys
from pathlib import Path

LOG = '/kaggle/working/notebook.log'

def sh(cmd, check=True):
    # Tee output to notebook.log: Kaggle's API log is often empty.
    with open(LOG, 'a') as log:
        log.write(f'$ {cmd}\n')
        p = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        log.write(p.stdout + f'[exit {p.returncode}]\n')
    print('$', cmd, '\n', p.stdout[-3000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')

def find_one(pattern):
    hits = glob.glob(pattern, recursive=True)
    assert hits, f'not found: {pattern}'
    return Path(hits[0])

REPO = Path('/kaggle/working/repo'); REPO.mkdir(exist_ok=True)
tgz = glob.glob('/kaggle/input/**/g4swe-code.tgz', recursive=True)
if tgz and Path(tgz[0]).is_file():
    sh(f'tar xzf {tgz[0]} -C {REPO}')
else:  # Kaggle sometimes unpacks archives into a folder
    src = find_one('/kaggle/input/**/src/g4swe/run_eval.py').parents[2]
    sh(f'cp -r {src}/. {REPO}/')
os.chdir(REPO)
WHEELHOUSE = find_one('/kaggle/input/**/gemma-4-developer-agent-wheelhouse').resolve()
DATA_DIR = find_one('/kaggle/input/**/gemma-4-developer-agent/tasks.jsonl').parent
print('wheelhouse:', WHEELHOUSE, '| data:', DATA_DIR)
os.environ['PYTHONPATH'] = 'src'

In [ ]:
import shutil
for pth in glob.glob('/usr/local/lib/python*/*-packages/*cutlass*.pth'):
    os.unlink(pth)
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(exist_ok=True)
for w in WHEELHOUSE.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
sh('pip install -q --no-deps --force-reinstall ' + ' '.join(sorted(str(w) for w in tmp_whl.glob('*.whl'))))
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')

In [ ]:
import shutil, traceback
import torch
from adk_submission import VllmConfig, VllmServer

MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
server = None
try:
    MODEL_PATH = find_one(f'/kaggle/input/models/**/{MODEL}/*/config.json').parent
    gpus = torch.cuda.device_count()
    tp = 4 if gpus >= 4 else (2 if gpus >= 2 else 1)
    server = VllmServer(VllmConfig(
        model=str(MODEL_PATH), port=8000, host='127.0.0.1',
        tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
        dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
        gpu_memory_utilization=0.90, enable_auto_tool_choice=True,
        scheduler_reserve_full_isl=False, tensor_parallel_size=tp, startup_timeout=60 * 20,
        extra_args=['--served-model-name', MODEL,  # agent YAML `model:` must equal this name
                    '--max-num-seqs', '8'],  # one agent at a time; vLLM's default 256 OOMs 31B on 4x L4
    ))
    server.start()
    print('vLLM up:', server.base_url, 'tp =', tp, 'gpus =', gpus)
    open(LOG, 'a').write(f'vLLM up: {server.base_url} tp={tp} gpus={gpus} model={MODEL_PATH}\n')
except Exception:
    # Python errors here never reach notebook.log otherwise; record them plus what the inputs look like.
    open(LOG, 'a').write('START_VLLM FAILED\n' + traceback.format_exc())
    sh('find /kaggle/input -maxdepth 6 -path "*models*" | head -40', check=False)
    raise
finally:
    if server is not None and getattr(server, 'log_path', None):
        shutil.copy(server.log_path, '/kaggle/working/vllm.log')
sh(f'python scripts/make_agents.py --model {MODEL}')

In [ ]:
# Configs k0, k1, k2, k3, k4, k5 on experiments/pilot_tasks.txt, one vLLM server. Results are packed after every
# config, so finished configs survive if the session hits its time limit.
for k in ['k0', 'k1', 'k2', 'k3', 'k4', 'k5']:
    sh(f'python -m g4swe.run_eval --mode agent --backend local --agent agents/{k} '
       f'--data-dir {DATA_DIR} --task-file experiments/pilot_tasks.txt --max-time-minutes 15', check=False)
    sh('tar czf /kaggle/working/results.tgz results')

In [ ]:
import json
sh('tar czf /kaggle/working/results.tgz results')
for f in sorted(glob.glob('results/*/runs.jsonl')):
    rows = [json.loads(line) for line in open(f)]
    print(f, sum(r['resolved'] for r in rows), '/', len(rows))
    for r in rows:
        print('   ', r)